# Task 3 - Advanced Models (Race To The Top)

This notebook implements advanced classical machine learning models that are particularly powerful on high-dimensional sparse data:
1. **LightGBM**: Fast, gradient boosted trees.
2. **Complement Naive Bayes**: A blazing fast NB model designed to correct the severe assumptions of standard NB for imbalanced text.
3. **LinearSVC**: The best performing model from the previous notebook (calibrated for probabilities).
4. **Soft Voting Ensemble**: An ensemble averaging the predicted probabilities of all models to achieve maximum Kaggle performance.

Models tried and key hyperparameters:
- Shared text features (`TfidfVectorizer` for all models): max_features=50000, ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True.
- `LightGBM`: n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1.
- `LinearSVC` + `CalibratedClassifierCV`: LinearSVC(C=1.0, random_state=42), calibration method='sigmoid', cv=3.
- `ComplementNB`: alpha=0.1.
- `VotingClassifier` ensemble: estimators=[ComplementNB, Calibrated LinearSVC, LightGBM], voting='soft', n_jobs=-1.

In [1]:
# Section 0 - Imports & Config
import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import VotingClassifier
import lightgbm as lgb

warnings.filterwarnings('ignore')

CONFIG = {
    'TRAIN_PATH': 'dataset/train.csv',
    'TEST_PATH': 'dataset/test.csv',
    'SUBMISSIONS_DIR': 'submissions',
    'RANDOM_STATE': 42,
    'VAL_SIZE': 0.15,
    # Shared TF-IDF representation used across all compared models below.
    'TFIDF': {
        'max_features': 50000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.95,
        'sublinear_tf': True,
    }
}

os.makedirs(CONFIG['SUBMISSIONS_DIR'], exist_ok=True)
np.random.seed(CONFIG['RANDOM_STATE'])
print('Config loaded.')

Config loaded.


## Section 1 - Load Data & Split

In [3]:
if not os.path.exists(CONFIG['TRAIN_PATH']):
    raise FileNotFoundError("Please place train.csv and test.csv in the dataset/ directory!")

train_df = pd.read_csv(CONFIG['TRAIN_PATH'], sep='\t')
test_df = pd.read_csv(CONFIG['TEST_PATH'], sep='\t')

train_texts = train_df['abstract'].astype(str)
train_labels = train_df['label_id'].astype(int)
test_texts = test_df['abstract'].astype(str)
test_ids = test_df['id'].astype(int)

X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_texts, train_labels, test_size=CONFIG['VAL_SIZE'], 
    random_state=CONFIG['RANDOM_STATE'], stratify=train_labels
)

print(f'Train split: {len(X_train_text)} samples')
print(f'Val split:   {len(X_val_text)} samples')

Train split: 118282 samples
Val split:   20874 samples


## Section 2 - TF-IDF Vectorization

In [4]:
print('Fitting TF-IDF...')
t0 = time.time()
vec = TfidfVectorizer(**CONFIG['TFIDF'])
X_train_vec = vec.fit_transform(X_train_text)
X_val_vec = vec.transform(X_val_text)
X_test_vec = vec.transform(test_texts)
print(f'TF-IDF Done in {time.time()-t0:.2f}s. Shape: {X_train_vec.shape}')

Fitting TF-IDF...
TF-IDF Done in 44.92s. Shape: (118282, 50000)


## Section 3 - Define Models for Ensemble 
We will define three strong standalone models: LightGBM, Calibrated LinearSVC, and ComplementNB.

In [5]:
# 1. LightGBM (tree baseline to compare against linear models on sparse TF-IDF)
lgbm_clf = lgb.LGBMClassifier(
    n_estimators=100,  # enough boosting rounds for a strong baseline
    learning_rate=0.1, # default-stable shrinkage for first pass
    random_state=CONFIG['RANDOM_STATE'],
    n_jobs=-1
)

# 2. LinearSVC (wrapped so we can use predict_proba in soft voting)
svc = LinearSVC(C=1.0, random_state=CONFIG['RANDOM_STATE'])  # core margin strength
calibrated_svc = CalibratedClassifierCV(estimator=svc, method='sigmoid', cv=3)  # 3-fold probability calibration

# 3. Complement Naive Bayes (fast, robust text baseline)
cnb_clf = ComplementNB(alpha=0.1)  # smoothing strength for rare terms

# Let's check them individually on the validation set
models = {
    'ComplementNB': cnb_clf,
    'LinearSVC (Calibrated)': calibrated_svc,
    'LightGBM': lgbm_clf
}

for name, model in models.items():
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_val_vec)
    f1 = f1_score(y_val, preds, average='macro')
    print(f"{name} Val Macro F1: {f1:.4f} (Time: {time.time()-t0:.2f}s)")

Training ComplementNB...
ComplementNB Val Macro F1: 0.4598 (Time: 0.91s)
Training LinearSVC (Calibrated)...
LinearSVC (Calibrated) Val Macro F1: 0.6577 (Time: 179.65s)
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 16.102305 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3582211
[LightGBM] [Info] Number of data points in the train set: 118282, number of used features: 49970
[LightGBM] [Info] Start training from score -2.884942
[LightGBM] [Info] Start training from score -4.224372
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -5.689362
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -3.984160
[LightGBM] [Info] 

## Section 4 - The Soft Voting Ensemble

In [6]:
print('Training Soft Voting Ensemble...')
t0 = time.time()
ensemble = VotingClassifier(
    estimators=[
        ('cnb', cnb_clf),
        ('svc', calibrated_svc),
        ('lgb', lgbm_clf)
    ],
    voting='soft',   # use probabilities to blend model confidence
    n_jobs=-1
)

ensemble.fit(X_train_vec, y_train)
ens_preds = ensemble.predict(X_val_vec)
ens_f1 = f1_score(y_val, ens_preds, average='macro')
print(f"Ensemble Val Macro F1: {ens_f1:.4f} (Time: {time.time()-t0:.2f}s)")

Training Soft Voting Ensemble...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 28.145192 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3582211
[LightGBM] [Info] Number of data points in the train set: 118282, number of used features: 49970
[LightGBM] [Info] Start training from score -2.884942
[LightGBM] [Info] Start training from score -4.224372
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -5.689362
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -3.984160
[LightGBM] [Info] Start training from score -2.856149
[LightGBM] [Info] Start training from score -3.329452
[LightGBM] [Info] Start training from score -2.971032
[LightGBM] [Info] Start training from score -4.644678
[LightGBM] [Info] Star

## Section 5 - Final Submissions Data Export

In [7]:
print('Generating full Kaggle test predictions...')
# LightGBM alone
lgbm_test_preds = lgbm_clf.predict(X_test_vec)
lgbm_sub = pd.DataFrame({'id': test_ids, 'label_id': lgbm_test_preds})
lgbm_sub.to_csv(os.path.join(CONFIG['SUBMISSIONS_DIR'], 'task3_lightgbm.csv'), index=False)

# Ensemble 
ens_test_preds = ensemble.predict(X_test_vec)
ens_sub = pd.DataFrame({'id': test_ids, 'label_id': ens_test_preds})
ens_sub.to_csv(os.path.join(CONFIG['SUBMISSIONS_DIR'], 'task3_ensemble.csv'), index=False)

print('Saved task3_lightgbm.csv and task3_ensemble.csv. Upload to Kaggle!')

Generating full Kaggle test predictions...
Saved task3_lightgbm.csv and task3_ensemble.csv. Upload to Kaggle!
